### Bibliotecas e Dependências

In [ ]:
pip install -U pymupdf4llm

In [19]:
import pandas as pd
import requests
import pymupdf4llm
import re
import os
import json
import unicodedata
import urllib3
from langdetect import detect
import time

Leitura do Arquivo

In [21]:
df = pd.read_csv(r'C:\Users\05646078199\Projetos\Projeto-Mestrado\Corpus\metadados_completos_sol_sbc.csv')

In [22]:
df["index"] = range(1, len(df) + 1)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29678 entries, 0 to 29677
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Title      29678 non-null  object
 1   Category   29678 non-null  object
 2   URL_Title  29678 non-null  object
 3   Authors    29678 non-null  object
 4   Event      29678 non-null  object
 5   Date       29678 non-null  object
 6   Box        29678 non-null  object
 7   Abstract   29677 non-null  object
 8   Keywords   29678 non-null  object
 9   Publisher  29678 non-null  object
 10  URL_Paper  29678 non-null  object
 11  index      29678 non-null  int64 
dtypes: int64(1), object(11)
memory usage: 2.7+ MB


In [41]:
df.to_csv(r'C:\Users\05646078199\Projetos\Projeto-Mestrado\Corpus\metadados_completos_sol_sbc.csv', index=False)

In [38]:
df_test = df[
    (df["index"] >= 1001) &
    (df["index"] <= 2000)
]

In [ ]:
df_01 =df[
    (df["index"] >= 2001) &
    (df["index"] <= 5000)
]

df_02 =df[
    (df["index"] >= 5001) &
    (df["index"] <= 8000)
]

df_03 =df[
    (df["index"] >= 8001) &
    (df["index"] <= 11000)
]

df_04 =df[
    (df["index"] >= 11001) &
    (df["index"] <= 14000)
]

df_05 =df[
    (df["index"] >= 14001) &
    (df["index"] <= 17000)
]


df_06 =df[
    (df["index"] >= 17001) &
    (df["index"] <= 20000)
]


df_07 =df[
    (df["index"] >= 21001) &
    (df["index"] <= 24000)
]


df_08 =df[
    (df["index"] >= 24001) &
    (df["index"] <= 27000)
]


df_09 =df[
    (df["index"] >= 27001)
]

In [39]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 1000 to 1999
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Title      1000 non-null   object
 1   Category   1000 non-null   object
 2   URL_Title  1000 non-null   object
 3   Authors    1000 non-null   object
 4   Event      1000 non-null   object
 5   Date       1000 non-null   object
 6   Box        1000 non-null   object
 7   Abstract   1000 non-null   object
 8   Keywords   1000 non-null   object
 9   Publisher  1000 non-null   object
 10  URL_Paper  1000 non-null   object
 11  index      1000 non-null   int64 
dtypes: int64(1), object(11)
memory usage: 101.6+ KB


In [ ]:
"""
1. Requisição para obtenção dos PDF - OK
2. Armazenamento do PDF Localmente - OK
3. Extração do conteúdo do PDF utilizando o pymupdf4llm armazenando o resultado em formato Markdown - OK
4. Identificação das seções e extração do conteúdo de cada seção - OK
5. Normalização do Texto 
6. Detecção do Idioma da Introdução ou Conclusão 
6. Armazenamento do conteúdo extraído em JSON (um arquivo JSON por artigo)
"""

In [27]:
def detect_article_language(sections):

    # tenta usar introdução primeiro
    for section in sections:

        title = section["title"].lower()

        if "introdu" in title:
            text = section["content"][:2000]

            try:
                return detect(text)
            except:
                pass

    # fallback: usa maior seção do artigo
    try:

        largest_section = max(
            sections,
            key=lambda s: len(s["content"])
        )

        return detect(largest_section["content"][:2000])

    except:
        return "unknown"

In [28]:
def save_article_json(article_data, output_path):

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            article_data,
            f,
            ensure_ascii=False,
            indent=4
        )

In [29]:
def normalize_text(text):

    # normalização unicode
    text = unicodedata.normalize("NFKC", text)

    # remove hifenização de quebra de linha
    text = re.sub(r'-\s*\n\s*', '', text)

    # transforma quebras simples em espaço
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)

    # remove múltiplos espaços
    text = re.sub(r'\s+', ' ', text)

    # remove espaços nas bordas
    text = text.strip()

    return text

In [30]:
def extract_sections(md_text):
    pattern = re.compile(r'^(#{1,6})\s+(.*)$', re.MULTILINE)

    matches = list(pattern.finditer(md_text))

    sections = []

    for i, match in enumerate(matches):
        level = len(match.group(1))

        # limpa markdown do título
        title = match.group(2).strip()
        title = re.sub(r'\*+', '', title).strip()

        start = match.end()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(md_text)

        content = md_text[start:end].strip()

        content = normalize_text(content)

        sections.append({
            "level": level,
            "title": title,
            "content": content
        })

    return sections

In [40]:
os.makedirs("pdfs", exist_ok=True)
os.makedirs("markdown", exist_ok=True)
os.makedirs("json", exist_ok=True)

# =========================================================
# CONFIGURAÇÕES
# =========================================================

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/pdf"
}

MAX_RETRIES = 1

failed_articles = []

total = len(df_test)

# =========================================================
# LOOP PRINCIPAL
# =========================================================

for index, row in df_test.iterrows():

    paper_id = f"sbc_{index}"

    pdf_path = f"pdfs/{paper_id}.pdf"
    md_path = f"markdown/{paper_id}.md"
    json_path = f"json/{paper_id}.json"

    print("\n" + "=" * 80)
    print(f"[{index + 1}/{total}] Processando {paper_id}")

    # =====================================================
    # EVITA REPROCESSAMENTO
    # =====================================================

    if os.path.exists(json_path):

        print("Artigo já processado.")
        continue

    try:

        # =================================================
        # URL PDF
        # =================================================

        url = row["URL_Paper"]

        pdf_url = url.replace("/view/", "/download/")

        print("URL PDF:", pdf_url)

        # =================================================
        # DOWNLOAD COM RETRY
        # =================================================

        response = None

        for attempt in range(MAX_RETRIES):

            try:

                print(f"Tentativa download {attempt + 1}/{MAX_RETRIES}")

                response = requests.get(
                    pdf_url,
                    headers=headers,
                    timeout=60,
                    verify=False
                )

                response.raise_for_status()

                # valida assinatura PDF
                if not response.content.startswith(b"%PDF"):
                    raise Exception("Arquivo baixado não é PDF")

                print("Download realizado com sucesso.")

                break

            except Exception as e:

                print(f"Falha na tentativa {attempt + 1}: {e}")

                if attempt == MAX_RETRIES - 1:
                    raise

                time.sleep(2)

        # =================================================
        # VALIDAÇÕES
        # =================================================

        if response is None:
            raise Exception("Resposta inválida")

        if len(response.content) == 0:
            raise Exception("PDF vazio")

        # =================================================
        # SALVA PDF
        # =================================================

        with open(pdf_path, "wb") as f:
            f.write(response.content)

        print("PDF salvo.")

        # =================================================
        # PDF -> MARKDOWN
        # =================================================

        md_text = pymupdf4llm.to_markdown(
            pdf_path,
            write_images=False
        )

        if not md_text.strip():
            raise Exception("Markdown vazio")

        with open(md_path, "w", encoding="utf-8") as f:
            f.write(md_text)

        print("Markdown salvo.")

        # =================================================
        # EXTRAÇÃO DE SEÇÕES
        # =================================================

        sections = extract_sections(md_text)

        if len(sections) == 0:
            raise Exception("Nenhuma seção encontrada")

        print(f"{len(sections)} seções encontradas.")

        # =================================================
        # DETECÇÃO DE IDIOMA
        # =================================================

        language = detect_article_language(sections)

        print("Idioma detectado:", language)

        # =================================================
        # ESTRUTURA FINAL
        # =================================================

        article_data = {
            "paper_id": paper_id,
            "title": row["Title"],
            "event": row["Event"],
            "authors": row["Authors"],
            "abstract_original": row["Abstract"],
            "url_paper": pdf_url,
            "language": language,
            "status": "success",
            "sections": sections
        }

        # =================================================
        # SALVA JSON
        # =================================================

        save_article_json(article_data, json_path)

        print("JSON salvo.")

        # =================================================
        # LIMPEZA OPCIONAL
        # =================================================

        # remove intermediários
        # descomente se quiser economizar espaço

        # os.remove(pdf_path)
        # os.remove(md_path)

    except Exception as e:

        error_message = str(e)

        print(f"Erro no artigo {paper_id}: {error_message}")

        failed_articles.append({
            "paper_id": paper_id,
            "url": pdf_url,
            "error": error_message
        })

# =========================================================
# SALVA LOG DE FALHAS
# =========================================================

with open("failed_articles.json", "w", encoding="utf-8") as f:

    json.dump(
        failed_articles,
        f,
        ensure_ascii=False,
        indent=4
    )

print("\nProcessamento finalizado.")
print(f"Falhas totais: {len(failed_articles)}")


[1001/1000] Processando sbc_1000
URL PDF: https://sol.sbc.org.br/index.php/semish/article/download/36797/36583
Tentativa download 1/1
Download realizado com sucesso.
PDF salvo.
Markdown salvo.
19 seções encontradas.
Idioma detectado: en
JSON salvo.

[1002/1000] Processando sbc_1001
URL PDF: https://sol.sbc.org.br/index.php/semish/article/download/36798/36584
Tentativa download 1/1
Download realizado com sucesso.
PDF salvo.
Markdown salvo.
17 seções encontradas.
Idioma detectado: en
JSON salvo.

[1003/1000] Processando sbc_1002
URL PDF: https://sol.sbc.org.br/index.php/semish/article/download/36799/36585
Tentativa download 1/1
Download realizado com sucesso.
PDF salvo.
Markdown salvo.
12 seções encontradas.
Idioma detectado: en
JSON salvo.

[1004/1000] Processando sbc_1003
URL PDF: https://sol.sbc.org.br/index.php/semish/article/download/36801/36587
Tentativa download 1/1
Download realizado com sucesso.
PDF salvo.
Markdown salvo.
16 seções encontradas.
Idioma detectado: en
JSON salvo.
